## Import Data

In [ ]:
import os
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter

def get_data(all_sessions_path, dt, gc_or_not=True):
    # 初始化为字典，用于按个体存储数据
    n_data_dict = {}
    f_data_dict_out = {} 
    neuron_ids_dict = {}
    global_id = 0

    for file in tqdm(all_sessions_path):
        print(file)
        
        # 提取文件夹名称并获取个体名字 (例如: 'MMELOIK')
        folder_name = os.path.basename(file)
        subject_name = folder_name.split('_')[0]
        
        # 如果这个个体还没有在字典中，则为他初始化空列表
        if subject_name not in n_data_dict:
            n_data_dict[subject_name] = []
            f_data_dict_out[subject_name] = []
            neuron_ids_dict[subject_name] = []

        try:
            # Chargement
            n_data = np.load(os.path.join(file, f'data_{dt}.npy'))
            f_data_raw = np.load(os.path.join(file, f'features_{dt}.npy'), allow_pickle=True)
            if gc_or_not:
                gc = np.load(os.path.join(file, 'good_clusters.npy'))
            else:
                gc = np.arange(len(n_data))
                
            unique_tones_path = r"C:\Users\PenPen\Desktop\Ferret\Data\Bohan\unique_tones\unique_tones.npy"
            try:
                unique_tones = np.load(unique_tones_path)
            except FileNotFoundError:
                print(f"Unique tones file not found at path: {unique_tones_path}")
                raise
                
            # Neurones valides
            n_data = n_data[gc, :].astype(float)

            # Construction DataFrame
            f_data_dict = {
                'Played_frequency': [],
                'Condition': [],
                'Block': [],
                'Frequency_changes': [],
                'Mock_frequency': [],
                'Mock_change': []
            }
            for item in f_data_raw:
                for key, value in item.items():
                    f_data_dict[key].append(value)
            f_data = pd.DataFrame(f_data_dict)

            # IDs des neurones
            N_neurons = len(n_data)
            neuron_ids = list(range(global_id, global_id + N_neurons))
            global_id += N_neurons
            neuron_ids_dict[subject_name].append(neuron_ids) # 存入对应个体的字典

            # --- mapping des fréquences vers pixels ---
            unique_tones_sorted = np.sort(unique_tones)
            pixels_sorted = np.linspace(0, 28, len(unique_tones_sorted))  # 28 cm

            # --- choisir Played ou Mock selon la condition ---
            positional_freq = np.where(
                f_data['Condition'].isin([0, -1]),
                f_data['Played_frequency'],
                f_data['Mock_frequency']
            )

            # --- interpolation pour toutes les fréquences inconnues ---
            positions = np.interp(positional_freq, unique_tones_sorted, pixels_sorted)

            # --- lissage ---
            pos_smooth = gaussian_filter(positions, sigma=10)

            # --- calcul de Speed_x ---
            speed_x = np.diff(pos_smooth)
            speed_x = np.append(0, speed_x)
            speed_x = np.abs(speed_x) * 100
            speed_x[~np.isfinite(speed_x)] = 0  # supprime NaN / inf
            f_data['Speed_x'] = speed_x
            # --- calcul de Sound_speed ---
            freq = np.array(f_data['Played_frequency'])
            freq_pos = np.interp(freq, unique_tones_sorted, pixels_sorted)
            freq_pos_smooth = gaussian_filter(freq_pos, sigma=10)
            sound_speed = np.diff(freq_pos_smooth)
            sound_speed = np.append(0, sound_speed)
            sound_speed = np.abs(sound_speed) * 100
            sound_speed[~np.isfinite(sound_speed)] = 0
            f_data['Sound_speed'] = sound_speed
            f_data['Position'] = pos_smooth
            f_data['Freq_position'] = freq_pos_smooth

            speed_bins = [0, 0.01, 1, 2, 3, 4, 5, 6, np.inf]
            f_data['Speed_bin'] = pd.cut(
                f_data['Speed_x'],
                bins=speed_bins,
                labels=False,
                include_lowest=True
            )
            acc_x = np.diff(speed_x, prepend=speed_x[0])
            f_data["Acc_x"] = acc_x

            # smooth n_data
            n_data_o = n_data - n_data.mean(axis=1, keepdims=True)
            n_data_smooth = gaussian_filter(n_data_o, sigma=1, axes=1)
            
            # Sauvegarde: 按个体追加到对应的字典中
            n_data_dict[subject_name].append(n_data_smooth)
            f_data_dict_out[subject_name].append(f_data)

        except Exception as e:
            print(f"Error for file {file}: {e}")
            
    # 返回字典而不是列表
    return n_data_dict, f_data_dict_out

In [ ]:
import os

base_path = r"C:\Users\PenPen\Desktop\Ferret\Data\Bohan"

# 获取所有 session 的路径
all_sessions_path = [
    os.path.join(base_path, name) 
    for name in os.listdir(base_path) 
    if os.path.isdir(os.path.join(base_path, name)) and "unique" not in name.lower()
]

dt = 0.005
# gc_or_not = True (if you only want the good clusters... keep it True)

# 现在返回的是字典结构
n_data_grouped, f_data_grouped = get_data(all_sessions_path, dt, gc_or_not=True)

# 打印出所有提取到的个体名称，确认是否正确
print("已加载数据的个体:", list(n_data_grouped.keys()))

# 示例：如何访问特定个体 (比如 MMELOIK) 的数据
# if 'MMELOIK' in n_data_grouped:
#     mmeloik_n_data = n_data_grouped['MMELOIK']  # 这是一个包含该个体所有 session n_data 的列表
#     mmeloik_f_data = f_data_grouped['MMELOIK']  # 这是一个包含该个体所有 session f_data 的列表
#     print(f"MMELOIK 共加载了 {len(mmeloik_n_data)} 个 sessions 的数据。")

## Set Up Essential Variables & Examining


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from tqdm import tqdm

# ==========================================
# 1. 拟合函数 (保持不变)
# ==========================================
def fit_lie_algebra_with_leak(r, x_dot, dt=0.005):
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0) 
    U_rot = r * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, r]) 
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dr_dt, rcond=None)
    weights = weights_T.T 
    
    J_unconstrained = weights[:, :N]
    L_leak = weights[:, N:]
    J_skew = 0.5 * (J_unconstrained - J_unconstrained.T)
    
    norm_total = np.linalg.norm(J_unconstrained)
    norm_skew = np.linalg.norm(J_skew)
    skewness_ratio = norm_skew / norm_total
    
    dr_dt_pred = U_rot @ J_skew.T + r @ L_leak.T
    r2 = r2_score(dr_dt.flatten(), dr_dt_pred.flatten())
    
    return J_skew, skewness_ratio, r2

# ==========================================
# 2. 遍历所有个体和 Session 进行分析
# ==========================================
all_results = []

# 这里的 n_data_grouped 和 f_data_grouped 是我们上一阶段得到的字典
for subject in n_data_grouped.keys():
    print(f"\n>>> Analyzing Subject: {subject}")
    
    subject_sessions_n = n_data_grouped[subject]
    subject_sessions_f = f_data_grouped[subject]
    
    # 遍历该个体的每一个 Session
    for idx, (n_data_session, f_df) in enumerate(zip(subject_sessions_n, subject_sessions_f)):
        
        r_full = n_data_session.T  # 转置为 (Time, Neurons)
        x_dot_full = f_df['Speed_x'].values
        conditions = f_df['Condition'].values
        
        # 处理不同的 Condition
        for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
            mask = (conditions == val)
            
            # 数据长度检查
            if np.sum(mask) < 500:
                continue
            
            # 执行拟合
            r_sub = r_full[mask]
            x_dot_sub = x_dot_full[mask]
            
            try:
                J_skew, skew_ratio, r2 = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
                
                # 保存结果到列表
                all_results.append({
                    'Subject': subject,
                    'Session_Idx': idx,
                    'Condition': label,
                    'Skewness_Ratio': skew_ratio,
                    'R2': r2,
                    'N_Neurons': r_sub.shape[1],
                    'J_Matrix': J_skew  # 如果需要存矩阵，可以保留这行
                })
            except Exception as e:
                print(f"Error in {subject} Session {idx} [{label}]: {e}")

# ==========================================
# 3. 结果汇总与展示
# ==========================================
results_df = pd.DataFrame(all_results)

# 打印整体摘要
print("\n--- Analysis Summary ---")
summary = results_df.groupby(['Subject', 'Condition'])[['Skewness_Ratio', 'R2']].mean()
print(summary)

# 示例：查看 HERCULE 的结果
# print(results_df[results_df['Subject'] == 'HERCULE'])

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 设置绘图风格
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. 绘制 Skewness Ratio 对比
sns.barplot(data=results_df, x='Subject', y='Skewness_Ratio', hue='Condition', ax=axes[0], palette='viridis')
axes[0].set_title('Skewness Ratio by Subject & Condition')
axes[0].set_ylim(0.8, 1.0) # 聚焦在高值区域
axes[0].set_ylabel('Skewness Ratio (Rotation Dominance)')

# 2. 绘制 R2 对比
sns.barplot(data=results_df, x='Subject', y='R2', hue='Condition', ax=axes[1], palette='magma')
axes[1].set_title('Model Explained Variance (R2)')
axes[1].set_ylabel('R-squared')

plt.tight_layout()
plt.show()

# Rotational Manifold (CEBRA)

## Raw Space

In [ ]:
import torch
import gc
import cebra
import numpy as np
import pandas as pd
from tqdm import tqdm

# ==========================================
# 1. 潜空间动力学拟合函数 (适配 z 空间)
# ==========================================
def fit_latent_generator(z, x_dot, dt=0.005):
    """
    在 CEBRA 降维后的潜空间 z (Time, 3) 上拟合旋转动力学
    """
    # 逻辑与之前的 fit_lie_algebra_with_leak 一致
    T, Dim = z.shape
    dz_dt = np.gradient(z, dt, axis=0) 
    
    U_rot = z * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, z]) 
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dz_dt, rcond=None)
    weights = weights_T.T 
    
    J_unconstrained = weights[:, :Dim]
    L_leak = weights[:, Dim:]
    J_skew = 0.5 * (J_unconstrained - J_unconstrained.T)
    
    norm_total = np.linalg.norm(J_unconstrained)
    norm_skew = np.linalg.norm(J_skew)
    lat_skew_ratio = norm_skew / (norm_total + 1e-9)
    
    dz_dt_pred = U_rot @ J_skew.T + z @ L_leak.T
    from sklearn.metrics import r2_score
    lat_r2 = r2_score(dz_dt.flatten(), dz_dt_pred.flatten())
    
    return lat_skew_ratio, lat_r2

# ==========================================
# 2. 遍历个体与 Session 执行 CEBRA
# ==========================================
cebra_results_list = []

for subject in n_data_grouped.keys():
    print(f"\n🧠 Processing CEBRA for Subject: {subject}")
    
    # 获取该个体的所有数据
    subject_sessions_n = n_data_grouped[subject]
    subject_sessions_f = f_data_grouped[subject]
    
    for idx, (n_data_session, f_df) in enumerate(zip(subject_sessions_n, subject_sessions_f)):
        print(f"  - Training Session {idx}...")
        
        # --- A. 显存清理 ---
        gc.collect()
        torch.cuda.empty_cache()
        
        # --- B. 数据准备 ---
        # CEBRA 输入通常为 (Time, Neurons)
        r_data = n_data_session.T.astype(np.float32)
        x_dot_data = f_df['Speed_x'].values.astype(np.float32)
        conditions = f_df['Condition'].values
        
        # --- C. 构建并训练模型 ---
        model = cebra.CEBRA(
            model_architecture='offset10-model',
            output_dimension=3,
            batch_size=512,        # 限制 batch 大小防止 5070 Ti OOM
            max_iterations=1000,   # 为了速度先设为 1000，稳定后可改为 2000
            distance='cosine',
            conditional='time_delta',
            device='cuda'
        )
        
        try:
            # 训练并变换
            model.fit(r_data, x_dot_data)
            z = model.transform(r_data)
            
            # --- 修改这里的打印逻辑 ---
            print(f"    ✅ Session {idx} Training Done. Evaluating Dynamics:")
            
            for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
                mask = (conditions == val)
                if np.sum(mask) < 500: 
                    print(f"      - [{label}]: Data too short, skipping.")
                    continue
                
                z_sub = z[mask]
                x_dot_sub = x_dot_data[mask]
                
                lat_skew, lat_r2 = fit_latent_generator(z_sub, x_dot_sub)
                
                # 在这里打印，就能看到每一个条件的实时结果
                print(f"      - [{label}] Latent R2: {lat_r2:.4f}, Skew: {lat_skew:.2%}")
                
                cebra_results_list.append({
                    'Subject': subject,
                    'Session_Idx': idx,
                    'Condition': label,
                    'Latent_Skewness': lat_skew,
                    'Latent_R2': lat_r2
                })
            
        except Exception as e:
            print(f"    ❌ Error in Session {idx}: {e}")

# ==========================================
# 3. 结果汇总
# ==========================================
cebra_df = pd.DataFrame(cebra_results_list)
print("\n--- Raw Space CEBRA Latent Analysis Summary ---")
print(cebra_df.groupby(['Subject', 'Condition'])[['Latent_Skewness', 'Latent_R2']].mean())

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
import cebra
from sklearn.metrics import r2_score
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

# ==========================================
# 1. 核心算法：潜空间动力学拟合 (保持用户逻辑)
# ==========================================
def fit_latent_generator(embedding, x_dot, dt=0.005):
    T, N = embedding.shape
    dz_dt = np.gradient(embedding, dt, axis=0) 
    
    U_rot = embedding * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, embedding])
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dz_dt, rcond=None)
    J_latent = weights_T.T[:, :N]
    L_leak = weights_T.T[:, N:]
    
    dz_dt_pred = U_rot @ J_latent.T + embedding @ L_leak.T
    r2 = r2_score(dz_dt.flatten(), dz_dt_pred.flatten())
    
    J_skew = 0.5 * (J_latent - J_latent.T)
    denom = np.linalg.norm(J_latent)
    skewness = np.linalg.norm(J_skew) / denom if denom > 1e-9 else 0
    
    return J_latent, skewness, r2

# ==========================================
# 2. 自动化批处理循环
# ==========================================
all_cebra_results = []

for subject in n_data_grouped.keys():
    print(f"\n" + "█" * 60)
    print(f"🚀 Processing Subject: {subject}")
    print("█" * 60)
    
    subject_n_list = n_data_grouped[subject]
    subject_f_list = f_data_grouped[subject]
    
    for idx, (n_raw, f_df) in enumerate(zip(subject_n_list, subject_f_list)):
        print(f"\n--- Session {idx} | Training CEBRA (Guided by Position) ---")
        
        # A. 显存清理
        gc.collect()
        torch.cuda.empty_cache()
        
        # B. 数据准备
        r_data = n_raw.T.astype(np.float32)
        pos_data = f_df['Position'].values.astype(np.float32)
        x_dot_data = f_df['Speed_x'].values.astype(np.float32)
        conditions = f_df['Condition'].values
        
        # C. CEBRA 降维
        model = cebra.CEBRA(
            model_architecture='offset10-model',
            output_dimension=3,
            batch_size=512,
            max_iterations=2000,
            distance='cosine',
            conditional='time_delta',
            device='cuda'
        )
        
        try:
            # 使用行为位置引导流形生成
            model.fit(r_data, pos_data)
            z = model.transform(r_data)
            
            # D. 高斯平滑 (魔咒破解器)
            z_smooth = gaussian_filter1d(z, sigma=5, axis=0)
            
            # E. 分 Condition 拟合动力学
            for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
                mask = (conditions == val)
                if np.sum(mask) < 500: continue
                
                z_sub = z_smooth[mask]
                x_dot_sub = x_dot_data[mask]
                
                # 拟合李代数
                J_lat, lat_skew, lat_r2 = fit_latent_generator(z_sub, x_dot_sub)
                
                print(f"  [{label}] R²: {lat_r2:.4f} | Skewness: {lat_skew:.2%}")
                
                all_cebra_results.append({
                    'Subject': subject,
                    'Session_Idx': idx,
                    'Condition': label,
                    'Skewness': lat_skew,
                    'R2': lat_r2,
                    'J_Matrix': J_lat
                })
                
        except Exception as e:
            print(f"  ❌ Error in Session {idx}: {e}")

# ==========================================
# 3. 统计汇总
# ==========================================
final_df = pd.DataFrame(all_cebra_results)
print("\n" + "=" * 45)
print("📊 FINAL TOPOLOGY-GUIDED SUMMARY")
print("=" * 45)
summary = final_df.groupby(['Subject', 'Condition'])[['R2', 'Skewness']].mean()
print(summary)

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
import cebra
from sklearn.metrics import r2_score
from scipy.ndimage import gaussian_filter1d

# ==========================================
# 1. 潜空间动力学拟合函数 
# ==========================================
def fit_latent_generator(embedding, x_dot, dt=0.005):
    T, N = embedding.shape
    dz_dt = np.gradient(embedding, dt, axis=0) 
    
    U_rot = embedding * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, embedding])
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dz_dt, rcond=None)
    J_latent = weights_T.T[:, :N]
    L_leak = weights_T.T[:, N:]
    
    dz_dt_pred = U_rot @ J_latent.T + embedding @ L_leak.T
    r2 = r2_score(dz_dt.flatten(), dz_dt_pred.flatten())
    
    J_skew = 0.5 * (J_latent - J_latent.T)
    denom = np.linalg.norm(J_latent)
    skewness = np.linalg.norm(J_skew) / denom if denom > 1e-9 else 0
    
    return J_latent, skewness, r2

# ==========================================
# 2. 自动化批处理：加入 Frequency_changes 过滤
# ==========================================
dynamic_window_results = []

for subject in n_data_grouped.keys():
    print(f"\n" + "█" * 60)
    print(f"🎯 Focusing on Stimulus-Driven Dynamics: {subject}")
    print("█" * 60)
    
    sub_n_list = n_data_grouped[subject]
    sub_f_list = f_data_grouped[subject]
    
    for idx, (n_raw, f_df) in enumerate(zip(sub_n_list, sub_f_list)):
        print(f"\n--- Session {idx} ---")
        
        # 显存清理
        gc.collect()
        torch.cuda.empty_cache()
        
        # 数据准备
        r_data = n_raw.T.astype(np.float32)
        pos_data = f_df['Position'].values.astype(np.float32)
        x_dot_data = f_df['Speed_x'].values.astype(np.float32)
        
        # 提取条件和频率变化标志
        conditions = f_df['Condition'].values
        freq_changes = f_df['Frequency_changes'].values
        
        # CEBRA 降维 (基于 Position 引导流形)
        model = cebra.CEBRA(
            model_architecture='offset10-model',
            output_dimension=3,
            batch_size=512,
            max_iterations=2000,
            distance='cosine',
            conditional='time_delta',
            device='cuda'
        )
        
        try:
            model.fit(r_data, pos_data)
            z = model.transform(r_data)
            
            # 高斯平滑
            z_smooth = gaussian_filter1d(z, sigma=5, axis=0)
            
            # --- 核心：定义 4 种子状态进行精细对比 ---
            analysis_groups = [
                ('Tracking_All', conditions == 0.0),
                ('Tracking_Moving (Freq_Change=1)', (conditions == 0.0) & (freq_changes == 1)),
                ('Playback_All', conditions == 1.0),
                ('Playback_Moving (Freq_Change=1)', (conditions == 1.0) & (freq_changes == 1))
            ]
            
            for label, mask in analysis_groups:
                # 降低一点数据量阈值，因为 freq_changes == 1 的时间窗口会比较短
                if np.sum(mask) < 150: 
                    print(f"  [{label}] ⚠️ Data too short ({np.sum(mask)} pts), skipping.")
                    continue
                
                z_sub = z_smooth[mask]
                x_dot_sub = x_dot_data[mask]
                
                # 拟合动力学
                J_lat, lat_skew, lat_r2 = fit_latent_generator(z_sub, x_dot_sub)
                
                print(f"  [{label}] R²: {lat_r2:.4f} | Skewness: {lat_skew:.2%}")
                
                dynamic_window_results.append({
                    'Subject': subject,
                    'Session_Idx': idx,
                    'Analysis_Window': label,
                    'Points_Count': np.sum(mask),
                    'Skewness': lat_skew,
                    'R2': lat_r2
                })
                
        except Exception as e:
            print(f"  ❌ Error in Session {idx}: {e}")

# ==========================================
# 3. 数据汇总与透视
# ==========================================
focused_df = pd.DataFrame(dynamic_window_results)

print("\n" + "=" * 60)
print("📊 DYNAMIC WINDOW COMPARISON SUMMARY")
print("=" * 60)
# 使用透视表更直观地对比 All vs Moving
summary_pivot = pd.pivot_table(
    focused_df, 
    values=['R2', 'Skewness'], 
    index=['Subject'], 
    columns=['Analysis_Window'], 
    aggfunc='mean'
)
print(summary_pivot)

### Ablation Study

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from tqdm import tqdm

# ==========================================
# 1. 核心算法：带 Leak 项的生成元拟合 (适配归一化空间)
# ==========================================
def fit_lie_algebra_normalized(r, x_dot, dt=0.005):
    """
    拟合公式: dr/dt = (x_dot * J_skew + L_leak) * r
    r 已经过 L2 归一化，位于单位球面上
    """
    T, N = r.shape
    # 计算切向速度 (速度矢量现在只代表方向的变化)
    dr_dt = np.gradient(r, dt, axis=0) 
    
    # 构建增强矩阵 [旋转驱动项, 衰减/偏移项]
    U_rot = r * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, r]) 
    
    # 最小二乘求解
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dr_dt, rcond=None)
    weights = weights_T.T # (N, 2N)
    
    J_unconstrained = weights[:, :N]
    L_leak = weights[:, N:]
    
    # 提取纯旋转分量 (反对称矩阵)
    J_skew = 0.5 * (J_unconstrained - J_unconstrained.T)
    
    # 计算指标
    norm_total = np.linalg.norm(J_unconstrained)
    norm_skew = np.linalg.norm(J_skew)
    skewness_ratio = norm_skew / (norm_total + 1e-9)
    
    # 计算预测值与 R2
    dr_dt_pred = U_rot @ J_skew.T + r @ L_leak.T
    r2 = r2_score(dr_dt.flatten(), dr_dt_pred.flatten())
    
    return skewness_ratio, r2

# ==========================================
# 2. 自动化批处理：剥离 Magnitude 分析
# ==========================================
normalization_results = []

for subject in n_data_grouped.keys():
    print(f"🌟 Analyzing Pure Directional Dynamics: {subject}")
    
    sub_n_list = n_data_grouped[subject]
    sub_f_list = f_data_grouped[subject]
    
    for idx, (n_raw, f_df) in enumerate(zip(sub_n_list, sub_f_list)):
        # 转置为 (Time, Neurons)
        r_matrix = n_raw.T 
        x_dot_data = f_df['Speed_x'].values
        conditions = f_df['Condition'].values
        
        # --- 核心消融操作：L2 Normalization ---
        # 计算每一时刻的群体矢量长度
        norms = np.linalg.norm(r_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1e-9 # 避免除零
        r_normalized = r_matrix / norms # 投影到单位球面
        
        # 分条件进行拟合
        for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
            mask = (conditions == val)
            if np.sum(mask) < 500: continue
            
            r_sub = r_normalized[mask]
            x_dot_sub = x_dot_data[mask]
            
            try:
                skew_ratio, r2 = fit_lie_algebra_normalized(r_sub, x_dot_sub)
                
                normalization_results.append({
                    'Subject': subject,
                    'Session_Idx': idx,
                    'Condition': label,
                    'Pure_Skewness': skew_ratio,
                    'Pure_R2': r2
                })
            except Exception as e:
                print(f"  Error in {subject} Session {idx}: {e}")

# ==========================================
# 3. 汇总与对比
# ==========================================
norm_df = pd.DataFrame(normalization_results)

print("\n" + "="*50)
print("📊 ABLATION STUDY: PURE DIRECTIONAL DYNAMICS")
print("="*50)
summary = norm_df.groupby(['Subject', 'Condition'])[['Pure_Skewness', 'Pure_R2']].mean()
print(summary)

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
import cebra
from sklearn.metrics import r2_score
from scipy.ndimage import gaussian_filter1d

# ==========================================
# 1. 拟合函数 (保持不变，支持 L 矩阵吸收对称方差)
# ==========================================
def fit_latent_generator(embedding, x_dot, dt=0.005):
    T, N = embedding.shape
    dz_dt = np.gradient(embedding, dt, axis=0) 
    U_rot = embedding * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, embedding])
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dz_dt, rcond=None)
    J_latent = weights_T.T[:, :N]
    L_leak = weights_T.T[:, N:]
    
    dz_dt_pred = U_rot @ J_latent.T + embedding @ L_leak.T
    r2 = r2_score(dz_dt.flatten(), dz_dt_pred.flatten())
    
    J_skew = 0.5 * (J_latent - J_latent.T)
    skewness = np.linalg.norm(J_skew) / np.linalg.norm(J_latent)
    
    return J_latent, skewness, r2

# ==========================================
# 2. 数据准备与归一化
# ==========================================
gc.collect()
torch.cuda.empty_cache()

session_idx = 0 
r_matrix_raw = n_data_s[session_idx].T
f_df = f_data_s[session_idx]
pos_array = f_df['Position'].values
x_dot_array = f_df['Speed_x'].values
conditions = f_df['Condition'].values

# 🌟 执行归一化消融：只保留方向信息
norms = np.linalg.norm(r_matrix_raw, axis=1, keepdims=True)
norms[norms == 0] = 1e-9
r_norm = (r_matrix_raw / norms).astype(np.float32)

# ==========================================
# 3. 训练共享流形 (确保坐标系一致)
# ==========================================
print("🚀 正在训练共享流形...")
model = cebra.CEBRA(
    model_architecture='offset10-model',
    output_dimension=3,
    batch_size=512,
    max_iterations=2000,
    distance='cosine',
    conditional='time_delta',
    device='cuda'
)
model.fit(r_norm, pos_array.astype(np.float32))
z = model.transform(r_norm)
z_smooth = gaussian_filter1d(z, sigma=5, axis=0)

# ==========================================
# 4. 条件拆分拟合
# ==========================================
results = []
print("\n" + "=" * 50)
print(f"✅ Session {session_idx} | Latent Dynamics Split Analysis")
print("=" * 50)

for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
    mask = (conditions == val)
    if np.sum(mask) < 500: continue
    
    # 提取对应条件的潜空间轨迹和速度
    z_sub = z_smooth[mask]
    x_dot_sub = x_dot_array[mask]
    
    # 拟合
    J_lat, lat_skew, lat_r2 = fit_latent_generator(z_sub, x_dot_sub)
    
    print(f"[{label} Mode]")
    print(f"🌟 Skewness Ratio : {lat_skew:.2%}")
    print(f"🌟 R² Score       : {lat_r2:.4f}")
    print("-" * 30)

print("=" * 50)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.stats import pearsonr

# 1. 把 CEBRA 的 3D 环投影到最主要的 2D 平面上 (寻找最佳的正圆视角)
pca = PCA(n_components=2)
# 注意我们要把中心对齐到原点，才能算角度
z_centered = z_smooth - np.mean(z_smooth, axis=0)
z_2d = pca.fit_transform(z_centered)

# 2. 提取神经流形的“内在相位” (Angle / Phase)
# 使用 arctan2 计算每个时间点在环上的角度，范围 [-pi, pi]
neural_phase = np.arctan2(z_2d[:, 1], z_2d[:, 0])

# 3. 解卷绕 (Unwrap)：防止角度从 pi 突然跳到 -pi 产生的巨大人工导数
neural_phase_unwrapped = np.unwrap(neural_phase)

# 4. 计算内在相位的角速度 (Neural Angular Velocity)
# 这一步对应你的方程左边：真正的状态变化率
d_theta_dt = np.gradient(neural_phase_unwrapped, 0.005)

# 5. 终极对决：流形角速度 vs 物理真实速度
# 因为我们不知道物理速度和流形相位的单位比例，我们计算皮尔逊相关系数
correlation, p_value = pearsonr(d_theta_dt, x_dot_data)

# --- 画图展示这个震撼的结果 ---
plt.figure(figsize=(10, 4))
time_axis = np.arange(2000) * 0.005 # 画前 10 秒的局部细节

# 对物理速度做一点缩放，只是为了能在同一张图里对齐显示趋势
scale_factor = np.std(d_theta_dt) / (np.std(x_dot_data) + 1e-9)

plt.plot(time_axis, d_theta_dt[:2000], label='Neural Manifold Angular Velocity ($\dot{\theta}$)', color='red', alpha=0.8)
plt.plot(time_axis, x_dot_data[:2000] * scale_factor, label='Physical Head Speed ($\dot{x}$)', color='black', alpha=0.6, linestyle='--')

plt.title(f'Continuous Attractor Validation\nPearson Correlation: {correlation:.4f} (p={p_value:.2e})', fontsize=14)
plt.xlabel('Time (s)')
plt.ylabel('Velocity (Normalized for visualization)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"🌟 神经流形角速度与物理速度的相关性 (Pearson r): {correlation:.4f}")

In [ ]:
# 引入 Leak 项的升级版回归
U_augmented = np.hstack([r_matrix * x_dot_array[:, np.newaxis], r_matrix])
J_augmented_T, _, _, _ = np.linalg.lstsq(U_augmented, np.gradient(r_matrix, 0.005, axis=0), rcond=None)

# 提取 J 和 leak
J_new = J_augmented_T.T[:, :14]  # 前 14 列是 J
Leak_matrix = J_augmented_T.T[:, 14:] # 后面是 Leak

# 重新计算反对称性
J_skew_new = 0.5 * (J_new - J_new.T)
print(f"引入 Leak 后，J 矩阵的反对称比例: {np.linalg.norm(J_skew_new, ord='fro') / np.linalg.norm(J_new, ord='fro'):.2%}")

## Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import r2_score

# ==========================================
# 0. 重新定义完整版拟合函数 (确保返回 4 个值)
# ==========================================
def fit_lie_algebra_with_leak_full(r, x_dot, dt=0.005):
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0) 
    
    U_rot = r * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, r]) 
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dr_dt, rcond=None)
    weights = weights_T.T
    
    J_unconstrained = weights[:, :N]
    L_leak = weights[:, N:]
    
    J_skew = 0.5 * (J_unconstrained - J_unconstrained.T)
    norm_total = np.linalg.norm(J_unconstrained)
    norm_skew = np.linalg.norm(J_skew)
    skewness_ratio = norm_skew / (norm_total + 1e-9)
    
    dr_dt_pred = U_rot @ J_skew.T + r @ L_leak.T
    r2 = r2_score(dr_dt.flatten(), dr_dt_pred.flatten())
    
    # 强制返回 4 个值: 原始J, 反对称J, 偏斜度, R2
    return J_unconstrained, J_skew, skewness_ratio, r2

# ==========================================
# 1. 封装：生成元 J 矩阵热图对比函数
# ==========================================
def plot_generator_heatmap(subject, session_idx, n_dict, f_dict):
    print("\n" + "=" * 60)
    print(f"🔥 Visualizing J Matrix: Subject [{subject}] | Session [{session_idx}]")
    print("=" * 60)
    
    try:
        r_full = n_dict[subject][session_idx].T
        f_df = f_dict[subject][session_idx]
        x_dot_full = f_df['Speed_x'].values
        conditions = f_df['Condition'].values
    except IndexError:
        print(f"⚠️ 找不到数据！个体 {subject} 没有 Session {session_idx}。")
        return

    J_results = {}

    # 拟合提取矩阵
    for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
        mask = (conditions == val)
        if np.sum(mask) > 500:
            r_sub = r_full[mask]
            x_dot_sub = x_dot_full[mask]
            
            # 使用上面刚刚定义的完整版函数
            J_unconstrained, J_skew, skewness, r2 = fit_lie_algebra_with_leak_full(r_sub, x_dot_sub)
            J_results[label] = {
                'J_raw': J_unconstrained,
                'skewness': skewness
            }

    if not J_results:
        print("⚠️ 数据量过少，无法画图。")
        return

    # 找到全局最大绝对值，统一色带范围 (vmin, vmax)
    all_J = np.concatenate([d['J_raw'].flatten() for d in J_results.values()])
    max_val = np.percentile(np.abs(all_J), 98) # 取 98% 分位数过滤极值

    # 画图
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    for i, label in enumerate(['Tracking', 'Playback']):
        ax = axes[i]
        if label not in J_results:
            ax.set_visible(False)
            continue
            
        J_raw = J_results[label]['J_raw']
        skewness = J_results[label]['skewness']
        
        # 使用 seaborn 热图
        sns.heatmap(J_raw, center=0, cmap='RdBu_r', 
                    vmin=-max_val, vmax=max_val, 
                    annot=False, square=True,    
                    cbar_kws={'label': 'Generator Value (Synaptic Weight)', 'shrink': 0.8},
                    ax=ax)
        ax.plot([0, J_raw.shape[1]], [0, J_raw.shape[0]], linestyle='--', color='black', linewidth=1.2, alpha=0.5)
        
        ax.set_title(f"{label} Condition\n(Natural Skewness: {skewness:.2%})", fontsize=14, pad=10)
        ax.set_xlabel("Neuron Index (Input)", fontsize=11)
        if i == 0:
            ax.set_ylabel("Neuron Index (Output)", fontsize=11)

    plt.suptitle(f"Lie Algebra Generator Matrix (J) Comparison\n{subject} - Session {session_idx}", 
                 fontsize=16, y=1.05, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ==========================================
# 调用示例：查看某个体的特定 Session
# ==========================================
plot_generator_heatmap(
    subject='MMELOIK', 
    session_idx=3, 
    n_dict=n_data_grouped, 
    f_dict=f_data_grouped
)

## Dynamical Systems Stability Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score

# ==========================================
# 1. 拟合函数 (确保返回完整的 J 矩阵)
# ==========================================
def fit_lie_algebra_with_leak(r, x_dot, dt=0.005):
    T, N = r.shape
    dr_dt = np.gradient(r, dt, axis=0) 
    
    U_rot = r * x_dot[:, np.newaxis]
    U_augmented = np.hstack([U_rot, r]) 
    
    weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dr_dt, rcond=None)
    weights = weights_T.T
    
    J_unconstrained = weights[:, :N]
    L_leak = weights[:, N:]
    
    J_skew = 0.5 * (J_unconstrained - J_unconstrained.T)
    norm_total = np.linalg.norm(J_unconstrained)
    norm_skew = np.linalg.norm(J_skew)
    skewness_ratio = norm_skew / (norm_total + 1e-9)
    
    dr_dt_pred = U_rot @ J_skew.T + r @ L_leak.T
    r2 = r2_score(dr_dt.flatten(), dr_dt_pred.flatten())
    
    return J_unconstrained, J_skew, skewness_ratio, r2

# ==========================================
# 2. 封装分析与画图函数
# ==========================================
def plot_eigenvalue_spectrum(subject, session_idx, n_dict, f_dict):
    print("\n" + "=" * 60)
    print(f"🔍 Analyzing Spectrum: Subject [{subject}] | Session [{session_idx}]")
    print("=" * 60)
    
    # 获取特定个体和 Session 的数据
    try:
        r_full = n_dict[subject][session_idx].T
        f_df = f_dict[subject][session_idx]
        x_dot_full = f_df['Speed_x'].values
        conditions = f_df['Condition'].values
    except IndexError:
        print(f"⚠️ 找不到对应的 Session 数据！个体 {subject} 只有 {len(n_dict[subject])} 个 Session。")
        return

    J_dict = {}

    # 循环提取并拟合
    for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
        mask = (conditions == val)
        if np.sum(mask) > 500:
            r_sub = r_full[mask]
            x_dot_sub = x_dot_full[mask]
            
            J, J_skew, skewness, r2 = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
            J_dict[label] = {
                'J': J,
                'skewness': skewness
            }

    if not J_dict:
        print("⚠️ 数据量过少，无法进行特征值分析。")
        return

    # ==========================================
    # 3. 绘制左右对比特征值复平面
    # ==========================================
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    colors = {'Tracking': '#E63946', 'Playback': '#457B9D'} # 使用更好看的红蓝配色

    # 找出全局最大特征值，用于统一坐标轴比例
    all_eigenvalues = np.concatenate([np.linalg.eigvals(d['J']) for d in J_dict.values()])
    max_val = np.max(np.abs(all_eigenvalues)) * 1.2

    for i, label in enumerate(['Tracking', 'Playback']):
        ax = axes[i]
        
        if label not in J_dict:
            ax.set_visible(False)
            continue
            
        J_matrix = J_dict[label]['J']
        skewness = J_dict[label]['skewness']
        
        # 计算特征值
        eigenvalues = np.linalg.eigvals(J_matrix)
        real_parts = np.real(eigenvalues)
        imag_parts = np.imag(eigenvalues)
        
        # 计算虚实比 (Imaginary / Real Ratio)
        mean_real = np.mean(np.abs(real_parts))
        mean_imag = np.mean(np.abs(imag_parts))
        vi_ratio = mean_imag / (mean_real + 1e-9)
        
        # 散点图绘制
        ax.scatter(real_parts, imag_parts, color=colors[label], s=50, alpha=0.6, edgecolors='white', linewidth=0.5)
        
        # 画十字坐标轴 (0,0)
        ax.axhline(0, color='black', linewidth=1.5, zorder=1)
        ax.axvline(0, color='black', linewidth=1.5, zorder=1)
        ax.grid(True, linestyle='--', alpha=0.3, zorder=0)
        
        # 设置统一的轴范围
        ax.set_xlim(-max_val, max_val)
        ax.set_ylim(-max_val, max_val)
        
        # 🌟 强制长宽比为 1:1，防止复平面被拉伸变形
        ax.set_aspect('equal', 'box')
        
        # 标签和标题
        ax.set_title(f"{label} Dynamics\nSkewness: {skewness:.2%} | Im/Re Ratio: {vi_ratio:.2f}", fontsize=13, pad=15)
        ax.set_xlabel("Real Part (Dissipation / Growth)", fontsize=11)
        if i == 0:
            ax.set_ylabel("Imaginary Part (Rotation / Oscillation)", fontsize=11)

    plt.suptitle(f"Eigenvalue Spectrum: {subject} - Session {session_idx}\nAligning to Y-axis indicates purely rotational dynamics", fontsize=15, y=1.05, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # ==========================================
    # 4. 打印统计报告
    # ==========================================
    for label in ['Tracking', 'Playback']:
        if label in J_dict:
            eig = np.linalg.eigvals(J_dict[label]['J'])
            real_mean = np.mean(np.abs(np.real(eig)))
            imag_mean = np.mean(np.abs(np.imag(eig)))
            ratio = imag_mean / (real_mean + 1e-9)
            print(f"  [{label}] 实部均值(耗散): {real_mean:.4f} | 虚部均值(旋转): {imag_mean:.4f} | 虚实比: {ratio:.2f}")

# ==========================================
# 调用示例：一键查看所有个体的 第 0 个 Session
# ==========================================
# 你可以修改 session_idx 或 subject 来查看特定的图表
for subject_name in n_data_grouped.keys():
    plot_eigenvalue_spectrum(
        subject=subject_name, 
        session_idx=0, 
        n_dict=n_data_grouped, 
        f_dict=f_data_grouped
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

# ==========================================
# 1. 批量提取 17 个 Session 的特征值
# ==========================================
all_results = []
min_points = 500

print(f"🚀 开始批量计算所有 Session 的特征值谱...")

for i in range(len(n_data_s)):
    r_full = n_data_s[i].T
    f_df = f_data_s[i]
    x_dot_full = f_df['Speed_x'].values
    conditions = f_df['Condition'].values
    
    session_metrics = {'session_id': i}
    valid_session = True
    
    # 循环提取 Tracking (0.0) 和 Playback (1.0)
    for val, cond_name in [(0.0, 'Tracking'), (1.0, 'Playback')]:
        mask = (conditions == val)
        if np.sum(mask) < min_points:
            valid_session = False  # 如果某个 Condition 数据不够，直接丢弃该 Session
            break
            
        r_sub = r_full[mask]
        x_dot_sub = x_dot_full[mask]
        
        try:
            # 调用拟合函数，提取原版 J 矩阵（J_unconstrained）
            J_raw, J_skew, skew_ratio, r2 = fit_lie_algebra_with_leak(r_sub, x_dot_sub)
            
            # 计算特征值的实部与虚部均值
            eigvals = np.linalg.eigvals(J_raw)
            real_mean = np.mean(np.abs(np.real(eigvals)))
            imag_mean = np.mean(np.abs(np.imag(eigvals)))
            
            session_metrics[f'{cond_name}_real'] = real_mean
            session_metrics[f'{cond_name}_imag'] = imag_mean
            
        except Exception as e:
            print(f"❌ Session {i} {cond_name} 拟合失败: {e}")
            valid_session = False
            break
            
    # 只有同时拥有 Tracking 和 Playback 的 Session 才会被记录
    if valid_session:
        all_results.append(session_metrics)

df_eig = pd.DataFrame(all_results)
print(f"\n✅ 成功提取 {len(df_eig)} 个配对 Session 的数据！")

# ==========================================
# 2. 统计检验 (Paired T-test)
# ==========================================
t_real, p_real = ttest_rel(df_eig['Tracking_real'], df_eig['Playback_real'])
t_imag, p_imag = ttest_rel(df_eig['Tracking_imag'], df_eig['Playback_imag'])

print("-" * 50)
print(f"【统计检验结果】")
print(f"Real Part (耗散) 差异 P值: {p_real:.4e}")
print(f"Imag Part (旋转) 差异 P值: {p_imag:.4e}")
print("-" * 50)

# ==========================================
# 3. 论文级可视化 (Figure 4: Slope Plots)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# --- 左图：实部 (Real Part / Dissipation) ---
ax1 = axes[0]
for idx in df_eig.index:
    ax1.plot(['Tracking', 'Playback'], 
             [df_eig.loc[idx, 'Tracking_real'], df_eig.loc[idx, 'Playback_real']], 
             color='gray', alpha=0.4, marker='o')

# 叠加均值和误差条
sns.pointplot(data=df_eig[['Tracking_real', 'Playback_real']], 
              color='red', markers='D', scale=1.5, ax=ax1)
ax1.set_xticklabels(['Tracking', 'Playback'])
ax1.set_title(f"Real Part (System Dissipation)\nPaired p-value: {p_real:.4e}", fontsize=14)
ax1.set_ylabel("Mean Absolute Real Part")
ax1.grid(axis='y', alpha=0.3)

# --- 右图：虚部 (Imaginary Part / Rotation) ---
ax2 = axes[1]
for idx in df_eig.index:
    ax2.plot(['Tracking', 'Playback'], 
             [df_eig.loc[idx, 'Tracking_imag'], df_eig.loc[idx, 'Playback_imag']], 
             color='gray', alpha=0.4, marker='o')

# 叠加均值和误差条
sns.pointplot(data=df_eig[['Tracking_imag', 'Playback_imag']], 
              color='blue', markers='D', scale=1.5, ax=ax2)
ax2.set_xticklabels(['Tracking', 'Playback'])
ax2.set_title(f"Imaginary Part (Rotational Core)\nPaired p-value: {p_imag:.4e}", fontsize=14)
ax2.set_ylabel("Mean Absolute Imaginary Part")
ax2.grid(axis='y', alpha=0.3)

plt.suptitle("Dynamical Signatures of Active Tracking vs. Passive Playback", fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

## Latent Phase-Coupling Analysis

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
import cebra
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr, wilcoxon
from sklearn.feature_selection import mutual_info_regression
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 生物学延迟对齐函数
# ==========================================
def get_best_mi_jitter(n_vel, p_vel, cond, target_val, window_samples=50):
    """
    在微小窗口内寻找最高互信息，校准神经信号与行为信号的延迟。
    """
    mask_full = (cond == target_val)
    if np.sum(mask_full) < 800:
        return 0.0, 0.0
        
    n_seg = n_vel[mask_full]
    p_seg = p_vel[mask_full]
    
    best_mi = -1.0
    best_r = 0.0
    
    # 尝试不同的延迟 (Shifts)
    for shift in range(-window_samples, window_samples + 1):
        if shift > 0:
            n_tmp, p_tmp = n_seg[shift:], p_seg[:-shift]
        elif shift < 0:
            n_tmp, p_tmp = n_seg[:shift], p_seg[abs(shift):]
        else:
            n_tmp, p_tmp = n_seg, p_seg
            
        # 仅在运动显著的时段计算
        v_thresh = np.std(p_tmp) * 0.1
        m = (np.abs(p_tmp) > v_thresh)
        
        if np.sum(m) > 500:
            # 随机采样以加速 MI 计算
            sample_size = min(1500, np.sum(m))
            idx = np.random.choice(np.where(m)[0], sample_size, replace=False)
            curr_mi = mutual_info_regression(p_tmp[idx].reshape(-1, 1), n_tmp[idx])[0]
            if curr_mi > best_mi:
                best_mi = curr_mi
                best_r = np.abs(pearsonr(n_tmp[m], p_tmp[m])[0])
                
    return max(0.0, best_mi), best_r

# ==========================================
# 2. 自动化批处理：潜空间相位耦合
# ==========================================
coupling_results = []

for subject in n_data_grouped.keys():
    print(f"\n" + "🌀" * 30)
    print(f"Refined Phase Analysis: {subject}")
    print("🌀" * 30)
    
    sub_n = n_data_grouped[subject]
    sub_f = f_data_grouped[subject]
    
    for idx, (n_raw, f_df) in enumerate(zip(sub_n, sub_f)):
        torch.cuda.empty_cache()
        gc.collect()
        
        try:
            # A. 数据预处理 (L2 归一化)
            r_v = n_raw.T.astype(np.float32)
            pos_v = f_df['Position'].values.astype(np.float32)
            cond_v = f_df['Condition'].values
            
            norms = np.linalg.norm(r_v, axis=1, keepdims=True)
            r_n = r_v / (norms + 1e-9)
            
            # B. CEBRA 降维提取流形
            model = cebra.CEBRA(model_architecture='offset10-model', batch_size=512, 
                                output_dimension=3, max_iterations=1500, 
                                distance='cosine', conditional='time_delta', device='cuda')
            model.fit(r_n, pos_v)
            z = model.transform(r_n)
            
            # C. 提取相位动力学 (Phase Dynamics)
            z_s = gaussian_filter1d(z, sigma=15, axis=0) # 平滑
            z_2d = PCA(n_components=2).fit_transform(z_s) # 投影到主要旋转平面
            z_c = z_2d - np.mean(z_2d, axis=0)
            
            # 计算神经旋转速度 (Angular Velocity)
            theta = np.unwrap(np.arctan2(z_c[:, 1], z_c[:, 0]))
            n_vel = np.gradient(theta, 0.005)
            
            # 计算物理位置速度
            p_vel = np.gradient(np.unwrap(np.deg2rad(pos_v * (360/28))), 0.005) # 将cm映射回角度
            
            # D. 计算两个 Condition 下的耦合强度
            for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
                mi, r_val = get_best_mi_jitter(n_vel, p_vel, cond_v, val)
                
                if mi > 0.0:
                    coupling_results.append({
                        'Subject': subject,
                        'Session': idx,
                        'Condition': label,
                        'MI': mi,
                        'Pearson_R': r_val
                    })
            
            print(f"  ✅ Session {idx} Coupling Analysis Done.")
            
        except Exception as e:
            print(f"  ⚠️ Session {idx} Skipped: {e}")

# ==========================================
# 3. 跨个体统计检验
# ==========================================
df_coupling = pd.DataFrame(coupling_results)

print("\n" + "="*50)
print("📊 最终统计报告：神经-行为相位耦合 (MI)")
print("="*50)

# 计算 Tracking 和 Playback 的配对对比
pivot_mi = df_coupling.pivot_table(index=['Subject', 'Session'], columns='Condition', values='MI').dropna()

if not pivot_mi.empty:
    stat, p_val = wilcoxon(pivot_mi['Tracking'], pivot_mi['Playback'])
    print(pivot_mi.mean())
    print(f"\n⚖️ Wilcoxon 配对检验 (Tracking vs Playback):")
    print(f"   P-value = {p_val:.4e}")
    if p_val < 0.05:
        print("   结论: Tracking 条件下的耦合显著强于 Playback。")
    else:
        print("   结论: 两种条件下耦合强度无显著差异。")
else:
    print("有效配对数据不足。")

## Latent Physical Phase-Coupling Analysis

In [ ]:
import gc
import torch
import numpy as np
import pandas as pd
import cebra
from sklearn.decomposition import PCA
from scipy.ndimage import gaussian_filter1d
from scipy.stats import pearsonr, wilcoxon
from sklearn.feature_selection import mutual_info_regression
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 生物学延迟对齐函数 (Jittering)
# ==========================================
def get_best_mi_jitter(n_vel, p_vel, cond, target_val, window_samples=50):
    """
    在微小窗口内寻找最高互信息，校准神经信号与行为信号的生理延迟。
    """
    mask_full = (cond == target_val)
    if np.sum(mask_full) < 800:
        return 0.0, 0.0
        
    n_seg = n_vel[mask_full]
    p_seg = p_vel[mask_full]
    
    best_mi = -1.0
    best_r = 0.0
    
    # 尝试不同的延迟 (Shifts)
    for shift in range(-window_samples, window_samples + 1):
        if shift > 0:
            n_tmp, p_tmp = n_seg[shift:], p_seg[:-shift]
        elif shift < 0:
            n_tmp, p_tmp = n_seg[:shift], p_seg[abs(shift):]
        else:
            n_tmp, p_tmp = n_seg, p_seg
            
        # 仅在物理运动显著的时段计算 (过滤静止噪音)
        v_thresh = np.std(p_tmp) * 0.1
        m = (np.abs(p_tmp) > v_thresh)
        
        if np.sum(m) > 500:
            sample_size = min(1500, np.sum(m))
            idx = np.random.choice(np.where(m)[0], sample_size, replace=False)
            curr_mi = mutual_info_regression(p_tmp[idx].reshape(-1, 1), n_tmp[idx])[0]
            if curr_mi > best_mi:
                best_mi = curr_mi
                best_r = np.abs(pearsonr(n_tmp[m], p_tmp[m])[0])
                
    return max(0.0, best_mi), best_r

# ==========================================
# 2. 潜空间物理相位耦合主循环
# ==========================================
print("🚀 启动: 潜空间物理相位耦合分析 (Latent Physical Phase-Coupling Analysis)")
coupling_results = []

for subject in n_data_grouped.keys():
    print(f"\n🌀 正在处理个体: {subject}")
    
    sub_n = n_data_grouped[subject]
    sub_f = f_data_grouped[subject]
    
    for idx, (n_raw, f_df) in enumerate(zip(sub_n, sub_f)):
        torch.cuda.empty_cache()
        gc.collect()
        
        try:
            # --- A. 数据提取与金钟罩容错 (NaN 过滤) ---
            r_raw_T = n_raw.T
            pos_raw = f_df['Position'].values
            cond_raw = f_df['Condition'].values
            
            v_mask = ~np.isnan(pos_raw)
            pos_v = pos_raw[v_mask].astype(np.float32)
            cond_v = cond_raw[v_mask]
            r_v = r_raw_T[v_mask].astype(np.float32)
            
            # --- B. 剥离幅度噪音 (L2 Normalization) ---
            norms = np.linalg.norm(r_v, axis=1, keepdims=True)
            r_n = r_v / (norms + 1e-9)
            
            # --- C. CEBRA 提取拓扑流形 ---
            model = cebra.CEBRA(model_architecture='offset10-model', batch_size=512, 
                                output_dimension=3, max_iterations=1500, 
                                distance='cosine', conditional='time_delta', device='cuda')
            model.fit(r_n, pos_v)
            z = model.transform(r_n)
            
            # --- D. PCA 投影提取神经相位动力学 ---
            z_s = gaussian_filter1d(z, sigma=15, axis=0) 
            z_2d = PCA(n_components=2).fit_transform(z_s) 
            z_c = z_2d - np.mean(z_2d, axis=0)
            
            theta = np.unwrap(np.arctan2(z_c[:, 1], z_c[:, 0]))
            n_vel = np.gradient(theta, 0.005)
            
            # --- E. 物理相位动力学完美映射 (关键步骤) ---
            # 28cm 轨道 -> 360度完整相位的转换
            p_vel = np.gradient(np.unwrap(np.deg2rad(pos_v * (360.0/28.0))), 0.005) 
            
            # --- F. 条件测算与校准 ---
            for val, label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
                mi, r_val = get_best_mi_jitter(n_vel, p_vel, cond_v, val)
                
                if mi > 0.0:
                    coupling_results.append({
                        'Subject': subject,
                        'Session': idx,
                        'Condition': label,
                        'MI': mi,
                        'Pearson_R': r_val
                    })
            
            print(f"  ✅ Session {idx} 耦合分析完成.")
            
        except Exception as e:
            print(f"  ⚠️ Session {idx} 跳过: {e}")

# ==========================================
# 3. 科研级跨个体统计检验
# ==========================================
df_coupling = pd.DataFrame(coupling_results)

print("\n" + "="*60)
print("📊 潜空间物理相位耦合摘要 (已校准延迟 & 物理维度映射)")
print("="*60)

# 使用透视表保证成对出现 (Paired data)
pivot_mi = df_coupling.pivot_table(index=['Subject', 'Session'], columns='Condition', values='MI').dropna()

if not pivot_mi.empty:
    stat, p_val = wilcoxon(pivot_mi['Tracking'], pivot_mi['Playback'])
    
    # 构建包含均值、标准差、标准误的完整表格
    stats_df = pivot_mi.describe()
    stats_df.loc['sem'] = pivot_mi.sem()
    
    print(stats_df.loc[['mean', 'std', 'sem']])
    print(f"\n⚖️ 配对 Wilcoxon 检验 P-value: {p_val:.4e}")
    
    mean_diff_pct = ((pivot_mi['Tracking'].mean() / pivot_mi['Playback'].mean()) - 1) * 100
    if p_val < 0.05:
        print(f"📈 结论: Tracking 耦合显著强于 Playback (差异 {mean_diff_pct:.2f}%).")
    else:
        print(f"🤝 结论: 两种条件下耦合强度无显著差异 (表现出高度的感觉表征不变性).")
else:
    print("未能获得足够的配对有效数据。")
print("="*60)

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

# 假设上一步得到的结果 DataFrame 名为 df_coupling
# 列名包含: 'Subject', 'Session', 'Condition', 'MI', 'Pearson_R'

# ==========================================
# 1. 数据汇总统计 (按 Condition 分组)
# ==========================================
print("📊 --- 神经相位耦合 (MI) 全局统计摘要 ---")
summary = df_coupling.groupby('Condition').agg({
    'MI': ['mean', 'std', 'sem'],
    'Pearson_R': ['mean', 'std']
}).round(4)
print(summary)
print("-" * 55)

# ==========================================
# 2. 配对统计检验 (Paired Test)
# ==========================================
# 【关键修改】使用 pivot_table 并把 Subject 和 Session 联合作为索引
pivot_df = df_coupling.pivot_table(
    index=['Subject', 'Session'], 
    columns='Condition', 
    values='MI'
).dropna()

if len(pivot_df) > 1:
    # Wilcoxon 符号秩检验
    stat, p_val = wilcoxon(pivot_df['Tracking'], pivot_df['Playback'])
    
    # 计算均值差异
    mi_diff = pivot_df['Tracking'].mean() - pivot_df['Playback'].mean()
    increase_pct = (mi_diff / pivot_df['Playback'].mean()) * 100

    print(f"⚖️ 统计显著性报告 (有效配对 N = {len(pivot_df)}):")
    print(f"   ↳ 互信息 (MI) 差异 P-value: {p_val:.4e}")
    
    if mi_diff > 0:
        print(f"   ↳ Tracking 相比 Playback MI 提升了: {mi_diff:.4f} (+{increase_pct:.1f}%)")
    else:
        print(f"   ↳ Tracking 相比 Playback MI 降低了: {abs(mi_diff):.4f} ({increase_pct:.1f}%)")
    
    # 结论判断
    if p_val < 0.05 and mi_diff > 0:
        print("   ✅ 结论：显著性达成！主动追踪显著增强了神经流形的相位锁定。")
    elif p_val < 0.05 and mi_diff < 0:
        print("   ✅ 结论：显著性达成！但在被动播放下耦合更强（需进一步解释）。")
    else:
        print("   🤝 结论：差异不显著。大脑展现出了高度的“感觉表征不变性”。")
else:
    print("⚠️ 匹配失败：有效配对 Session 数量不足，无法进行统计检验。")

# ==========================================
# 3. 逐个配对原始数据打印（方便核对）
# ==========================================
print("\n🔍 --- 各个体 & Session 详细配对记录 ---")
# 增加一列展示差异
pivot_df['Diff (TR-PB)'] = pivot_df['Tracking'] - pivot_df['Playback']
print(pivot_df.round(4).sort_index())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 定制化绘图：Tracking (左) vs Playback (右)
# ==========================================
# 确保 pivot_df 已经转换为长格式
plot_df = pivot_df.reset_index()

plt.figure(figsize=(8, 7))
sns.set_theme(style="ticks", context="talk")

# 为三只 ferret 分配不同的颜色，增加图表的信息维度
subject_colors = {
    'HERCULE': '#2A9D8F',  # 蓝绿色
    'MMELOIK': '#E76F51',  # 珊瑚红
    'NAPOLEON': '#E9C46A'  # 砂黄色
}

# 1. 画每条细线（注意这里的坐标对应：[0, 1] 对应 [Tracking, Playback]）
for idx, row in plot_df.iterrows():
    subj = row['Subject']
    plt.plot([0, 1], [row['Tracking'], row['Playback']], 
             marker='o', markersize=8, color=subject_colors[subj], 
             alpha=0.6, linewidth=1.5, zorder=1)

# 2. 计算全局均值与标准误
mean_tr = plot_df['Tracking'].mean()
mean_pb = plot_df['Playback'].mean()
sem_tr = plot_df['Tracking'].sem()
sem_pb = plot_df['Playback'].sem()

# 画全局均值粗线
plt.plot([0, 1], [mean_tr, mean_pb], marker='D', markersize=12, color='black', 
         linewidth=4, zorder=3, label='Population Mean')

# 画误差棒 (x=0 是 Tracking, x=1 是 Playback)
plt.errorbar(0, mean_tr, yerr=sem_tr, color='black', capsize=8, capthick=2, linewidth=3, zorder=4)
plt.errorbar(1, mean_pb, yerr=sem_pb, color='black', capsize=8, capthick=2, linewidth=3, zorder=4)

# 3. 美化坐标轴
# 调换 x 轴标签顺序
plt.xticks([0, 1], ['Tracking\n(Active)', 'Playback\n(Passive)'], fontsize=14, fontweight='bold')
plt.xlim(-0.2, 1.2)
plt.ylabel('Phase-Coupling Strength (Mutual Information)', fontsize=14, fontweight='bold')
plt.title('Active Sensing Enhances Neural-Stimulus Phase Locking', fontsize=16, fontweight='bold', pad=25)

# 4. 显著性标识 (p = 0.0128 -> *)
y_max = plot_df[['Playback', 'Tracking']].max().max() + 0.005
plt.plot([0, 0, 1, 1], [y_max-0.002, y_max, y_max, y_max-0.002], lw=1.5, c='black')
plt.text(0.5, y_max, '*\n(p=0.0128)', ha='center', va='bottom', color='black', fontsize=14, fontweight='bold')

# 5. 添加图例 (将图例移到右侧外部避免遮挡数据线)
handles = [plt.Line2D([0], [0], color=c, lw=2, marker='o', label=subj) for subj, c in subject_colors.items()]
handles.append(plt.Line2D([0], [0], color='black', lw=4, marker='D', label='Mean ± SEM'))
plt.legend(handles=handles, loc='upper left', frameon=False, fontsize=11, bbox_to_anchor=(1.02, 1))

sns.despine(trim=True)
plt.tight_layout()
plt.show()

In [ ]:
# import sys
# # 1. 强制在当前环境的 site-packages 中安装缺失依赖
# !{sys.executable} -m pip install literate-dataclasses requests tqdm --upgrade --force-reinstall

In [ ]:
import os
import sys

# 1. 尝试直接导入
try:
    import cebra
    print(f"✅ CEBRA 已就绪 (版本: {cebra.__version__})")
except ImportError:
    print("正在尝试强制补齐 CEBRA...")
    # 强制安装，但不允许它动你的 numpy
    %pip install cebra --no-deps
    import cebra
    print("✅ CEBRA 补丁安装成功！")

# 2. 检查环境
import torch
import numpy as np
print(f"NumPy 版本: {np.__version__}")
print(f"CUDA 状态: {torch.cuda.is_available()}")

# 3. 如果以上都通了，直接运行绘图测试
if torch.cuda.is_available():
    import cebra.models
    # 这里只是做一个最小化的模型测试，确保显卡能跑
    test_model = cebra.CEBRA(model_architecture='offset10-model', device='cuda', max_iterations=10)
    print("🚀 5070 Ti 握手成功，随时可以开始大规模渲染！")

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import cebra
import torch
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda'
session_idx = 0  

# ==========================================
# 1. 提取并“降采样”数据 (救显卡于水火)
# ==========================================
r_full = n_data_s[session_idx].T
f_df = f_data_s[session_idx]
pos_full = f_df['Position'].values
conditions = f_df['Condition'].values  

# 🌟 关键优化：每隔 10 个时间点取 1 个点 (Downsample by 10)
# 这能把 30万点 降到 3万点，大幅缩短训练时间，且不改变流形拓扑！
step = 10
r_subsampled = r_full[::step].astype(np.float32)
pos_subsampled = pos_full[::step].astype(np.float32)
cond_subsampled = conditions[::step]

# 过滤掉无效的离散点
valid_mask = ~np.isnan(pos_subsampled)
r_valid = r_subsampled[valid_mask]
pos_valid = pos_subsampled[valid_mask]
cond_valid = cond_subsampled[valid_mask]

# ==========================================
# 2. 训练 CEBRA (现在应该只要 1-2 分钟了)
# ==========================================
print(f"🚀 5070 Ti 启动！处理降采样后的 {len(r_valid)} 个数据点...")
model = cebra.CEBRA(
    model_architecture='offset10-model',
    temperature=1.5,
    batch_size=2048,        
    learning_rate=3e-4,
    output_dimension=3,     
    max_iterations=3000,    # 迭代次数稍微降一点，3000足够了
    distance='cosine',
    conditional='time_delta',
    device=device
)

model.fit(r_valid, pos_valid)
embedding = model.transform(r_valid)

# ==========================================
# 3. 数据拆分与可视化
# ==========================================
mask_track = (cond_valid == 0.0)
mask_play = (cond_valid == 1.0)

# 设置相同的绘图参数
fig = plt.figure(figsize=(16, 8))
all_max = np.max(np.abs(embedding), axis=0)

# --- Tracking ---
ax1 = fig.add_subplot(121, projection='3d')
sc1 = ax1.scatter(embedding[mask_track, 0], embedding[mask_track, 1], embedding[mask_track, 2], 
                  c=pos_valid[mask_track], cmap='twilight_shifted', s=2, alpha=0.5)
ax1.set_title("Active Tracking\n(Sensorimotor Loop Closed)", fontsize=16)
ax1.set_axis_off()
ax1.view_init(elev=30, azim=45)

# --- Playback ---
ax2 = fig.add_subplot(122, projection='3d')
sc2 = ax2.scatter(embedding[mask_play, 0], embedding[mask_play, 1], embedding[mask_play, 2], 
                  c=pos_valid[mask_play], cmap='twilight_shifted', s=2, alpha=0.5)
ax2.set_title("Passive Playback\n(Sensory Driven Only)", fontsize=16)
ax2.set_axis_off()
ax2.view_init(elev=30, azim=45)

# 统一坐标轴
for ax in [ax1, ax2]:
    ax.set_xlim(-all_max[0], all_max[0])
    ax.set_ylim(-all_max[1], all_max[1])
    ax.set_zlim(-all_max[2], all_max[2])

cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
fig.colorbar(sc1, cax=cbar_ax, label='Head Position (Degree)')

plt.suptitle(f"Shared Neural Manifold Topology (Session {session_idx})\n(Downsampled for Visualization)", fontsize=18, y=1.02)
plt.show()

In [ ]:
import seaborn as sns
from scipy.stats import ttest_ind

# 1. Map conditions from f_data_s to our summary table
# We take the first value of 'Condition' from each session's dataframe
conditions = [f_data_s[i]['Condition'].iloc[0] for i in range(len(f_data_s))]
df_summary['condition_name'] = ['Tracking' if c == 0.0 else 'Playback' for c in conditions]

# 2. Statistical Analysis (T-Test)
tracking_skew = df_summary[df_summary['condition_name'] == 'Tracking']['skewness']
playback_skew = df_summary[df_summary['condition_name'] == 'Playback']['skewness']
t_stat, p_val = ttest_ind(tracking_skew, playback_skew)

# 3. Visualization: The "Figure 4" of your Manuscript
plt.figure(figsize=(14, 6))

# Subplot A: Skew-Symmetry Comparison
plt.subplot(1, 2, 1)
sns.boxplot(x='condition_name', y='skewness', data=df_summary, palette='Set2', width=0.5)
sns.stripplot(x='condition_name', y='skewness', data=df_summary, color='black', alpha=0.5)
plt.title(f'Manifold Structural Integrity\n(p-value: {p_val:.4f})', fontsize=14)
plt.ylabel('Skew-Symmetry Ratio')

# Subplot B: Eigenvalue Ratio (Rotational Stability)
plt.subplot(1, 2, 2)
sns.violinplot(x='condition_name', y='vi_ratio', data=df_summary, inner='point', palette='Pastel1')
plt.title('Rotational Stability (Im/Re Ratio)', fontsize=14)
plt.ylabel('Imaginary / Real Ratio')

plt.tight_layout()
plt.show()

# 4. Print Group Statistics
print("\n--- [Condition-wise Comparison Report] ---")
report = df_summary.groupby('condition_name').agg({
    'skewness': ['mean', 'std'],
    'vi_ratio': ['mean', 'std']
})
print(report)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

# --- 1. Internal Session Analysis ---
all_results = []
dt = 0.005
min_points = 500

print(f"🚀 Processing {len(n_data_s)} sessions for internal comparison...")

for i in range(len(n_data_s)):
    r_full = n_data_s[i].T
    f_df = f_data_s[i]
    x_dot_full = f_df['Speed_x'].values
    conditions = f_df['Condition'].values # 0.0 is Tracking, 1.0 is Playback
    
    for cond_val, cond_label in [(0.0, 'Tracking'), (1.0, 'Playback')]:
        mask = (conditions == cond_val)
        if np.sum(mask) < min_points:
            continue
            
        r = r_full[mask]
        x_dot = x_dot_full[mask]
        
        try:
            # Fit Generator J
            U_augmented = np.hstack([r * x_dot[:, np.newaxis], r])
            dr_dt = np.gradient(r, dt, axis=0)
            weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dr_dt, rcond=None)
            J = weights_T.T[:, :r.shape[1]]
            
            # Metrics
            J_skew = 0.5 * (J - J.T)
            skewness = np.linalg.norm(J_skew) / np.linalg.norm(J)
            
            all_results.append({
                'session_id': i,
                'condition': cond_label,
                'skewness': skewness
            })
        except:
            continue

# Create DataFrames
df_blocks = pd.DataFrame(all_results)
# Pivot for Paired Comparison
df_pivot = df_blocks.pivot(index='session_id', columns='condition', values='skewness').dropna()

# --- 2. Paired T-Test ---
t_stat, p_val = ttest_rel(df_pivot['Tracking'], df_pivot['Playback'])

# --- 3. Visualization (Slope Plot) ---
plt.figure(figsize=(7, 7))

# Plot individual session lines
for idx in df_pivot.index:
    plt.plot(['Tracking', 'Playback'], 
             [df_pivot.loc[idx, 'Tracking'], df_pivot.loc[idx, 'Playback']], 
             color='gray', alpha=0.4, marker='o', linewidth=1)

# Overlay Mean and Error Bars
sns.pointplot(x='condition', y='skewness', data=df_blocks, 
              color='red', markers='D', linestyles='-', scale=1.2)

plt.title(f'Internal Manifold Stability: Tracking vs. Playback\n(Paired T-test p-value: {p_val:.4f})', fontsize=14)
plt.ylabel('Skew-Symmetry Ratio (Dynamic Integrity)')
plt.ylim(0.7, 1.0) # Zoom in to see the high-quality range
plt.grid(axis='y', alpha=0.2)
plt.tight_layout()
plt.show()

print(f"✅ Comparison complete. Found {len(df_pivot)} sessions with valid paired data.")

## 3D

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import cebra
import torch
import numpy as np
import matplotlib.pyplot as plt

# 开启交互式后端，允许鼠标拖拽旋转
plt.switch_backend('qtagg')

# --- 1. 参数与全局降采样 ---
target_session = 10
step = 10  # 每 10 个点取 1 个，涵盖整个 Session，救显卡于水火

# 提取全部数据并降采样
r_full = n_data_s[target_session].T[::step].astype(np.float32)
pos_full = f_data_s[target_session]['Position'].values[::step].astype(np.float32)
cond_full = f_data_s[target_session]['Condition'].values[::step]

# 过滤 NaN 异常值
valid_mask = ~np.isnan(pos_full)
r_valid = r_full[valid_mask]
pos_valid = pos_full[valid_mask]
cond_valid = cond_full[valid_mask]

# 🌟 核心操作：L2 归一化，把点强行推到球面上，让拓扑环更清晰！
norms = np.linalg.norm(r_valid, axis=1, keepdims=True)
norms[norms == 0] = 1e-9
r_norm = r_valid / norms

# --- 2. 训练全局 CEBRA 共享流形 ---
print(f"🚀 正在训练 Session {target_session} 的共享流形...")
model = cebra.CEBRA(
    model_architecture='offset10-model',
    batch_size=2048,
    output_dimension=3,
    max_iterations=3000,
    distance='cosine',        # 找回流形拓扑的关键参数
    conditional='time_delta', # 找回流形拓扑的关键参数
    device='cuda'
)
model.fit(r_norm, pos_valid)
embedding = model.transform(r_norm)

# # --- 3. 交互式 3D 对比可视化 ---
# # 拆分数据
# mask_tr = (cond_valid == 0.0)
# mask_pb = (cond_valid == 1.0)

# fig = plt.figure(figsize=(16, 8))
# all_max = np.max(np.abs(embedding), axis=0)

# # 【左图：Tracking】
# ax1 = fig.add_subplot(121, projection='3d')
# sc1 = ax1.scatter(embedding[mask_tr, 0], embedding[mask_tr, 1], embedding[mask_tr, 2],
#                   c=pos_valid[mask_tr], cmap='twilight_shifted', s=5, alpha=0.6)
# ax1.set_title("Active Tracking", fontsize=16)
# ax1.set_axis_off()

# # 【右图：Playback】
# ax2 = fig.add_subplot(122, projection='3d')
# sc2 = ax2.scatter(embedding[mask_pb, 0], embedding[mask_pb, 1], embedding[mask_pb, 2],
#                   c=pos_valid[mask_pb], cmap='twilight_shifted', s=5, alpha=0.6)
# ax2.set_title("Passive Playback", fontsize=16)
# ax2.set_axis_off()

# # 强制统一坐标轴（这样两边的流形大小才一致，具有可比性）
# for ax in [ax1, ax2]:
#     ax.set_xlim(-all_max[0], all_max[0])
#     ax.set_ylim(-all_max[1], all_max[1])
#     ax.set_zlim(-all_max[2], all_max[2])
#     ax.set_box_aspect([1, 1, 1])

# # 添加颜色条
# cbar_ax = fig.add_axes([0.92, 0.25, 0.02, 0.5])
# fig.colorbar(sc1, cax=cbar_ax, label='Head Position (Degree)')

# plt.suptitle(f"Interactive Manifold Comparison: Session {target_session}\n(Drag to Rotate)", fontsize=18)
# plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

def analyze_all_sessions(n_data_list, f_data_list, dt=0.005):
    all_results = []
    
    print(f"Starting bulk analysis on {len(n_data_list)} sessions...")
    
    for i in range(len(n_data_list)):
        try:
            # 1. Extract data for the current session
            r = n_data_list[i].T
            x_dot = f_data_list[i]['Speed_x'].values
            
            # 2. Fit Generator J with Leak term (The most accurate variant)
            # Construct augmented matrix [r * x_dot, r]
            U_augmented = np.hstack([r * x_dot[:, np.newaxis], r])
            dr_dt = np.gradient(r, dt, axis=0)
            
            # Solve using Least Squares
            weights_T, _, _, _ = np.linalg.lstsq(U_augmented, dr_dt, rcond=None)
            weights = weights_T.T
            
            # Split J (Generator) and Leak (Diagonal decay)
            num_neurons = r.shape[1]
            J = weights[:, :num_neurons]
            
            # 3. Calculate Skew-Symmetry Ratio
            # Formula: J_skew = 0.5 * (J - J.T)
            J_skew = 0.5 * (J - J.T)
            skewness = np.linalg.norm(J_skew) / np.linalg.norm(J)
            
            # 4. Calculate Eigenvalue Spectrum (Imaginary/Real Ratio)
            eigvals = np.linalg.eigvals(J)
            imag_mean = np.mean(np.abs(np.imag(eigvals)))
            real_mean = np.mean(np.abs(np.real(eigvals)))
            ratio = imag_mean / (real_mean + 1e-9) # Avoid division by zero
            
            # Store results
            all_results.append({
                'session_id': i,
                'skewness': skewness,
                'vi_ratio': ratio,
                'data_points': len(x_dot)
            })
            print(f"✅ Session {i} complete: Skew-symmetry={skewness:.2%}, Im/Re ratio={ratio:.2f}")
            
        except Exception as e:
            print(f"❌ Session {i} analysis failed: {e}")
            
    return pd.DataFrame(all_results)

# ==========================================
# Execute Global Analysis
# ==========================================
df_summary = analyze_all_sessions(n_data_s, f_data_s)

# ==========================================
# Visualization (Cross-Sample Evidence Plot)
# ==========================================
plt.figure(figsize=(12, 5))

# Subplot 1: Distribution of Skewness
plt.subplot(1, 2, 1)
plt.boxplot(df_summary['skewness'], vert=True)
plt.scatter(np.ones(len(df_summary)), df_summary['skewness'], color='red', alpha=0.5)
plt.axhline(0.9, color='green', linestyle='--', label='90% Threshold')
plt.title('Distribution of Skew-Symmetry across Sessions')
plt.ylabel('Skewness Ratio')
plt.legend()

# Subplot 2: Imaginary vs Real Ratio
plt.subplot(1, 2, 2)
plt.bar(df_summary['session_id'], df_summary['vi_ratio'], color='skyblue')
plt.axhline(5, color='orange', linestyle='--', label='5:1 (Strongly Imaginary)')
plt.title('Imaginary/Real Ratio across Sessions')
plt.ylabel('Ratio (Imag / Real)')
plt.xlabel('Session ID')
plt.legend()

plt.tight_layout()
plt.show()

print("\n--- [Full Sample Summary Report] ---")
print(f"Mean Skew-Symmetry: {df_summary['skewness'].mean():.2%}")
print(f"Mean Imag/Real Ratio: {df_summary['vi_ratio'].mean():.2f}")

In [ ]:
import numpy as np
import cebra

# 1. 计算速度/方向 (Speed/Direction)
# 这里我们假设 f_data_s[session_idx]['Position'] 是位置数据
# 计算每一帧与上一帧的位置差
position_raw = f_data_s[session_idx]['Position'].values[:sample_size]
direction = np.diff(position_raw) 

# np.diff 会让数据少一个点，我们需要补齐，或者调整 neural_data
# 这里我们选择补齐一个点，假设第一帧的方向是 0
direction = np.concatenate([[0], direction])

# 2. 准备数据和模型 (这部分代码保持不变，确保使用 CUDA)
r_data = r_matrix[:sample_size, :].astype(np.float32)

model = cebra.CEBRA(
    model_architecture='offset10-model',
    batch_size=4096,
    learning_rate=3e-4,
    output_dimension=3,
    max_iterations=5000, 
    device='cuda'  # 再次确认使用了 CUDA
)

print("🚀 正在重新计算流形...")
model.fit(r_data, np.zeros(sample_size)) # 这里 CEBRA 训练不需要 label，我们用全零代替
embedding = model.transform(r_data)

# 3. 使用方向重新着色绘图
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# 关键改动：c=direction
# cmap='seismic' 是一个非常适合表示正负（比如速度）的色带：蓝色表示负值，红色表示正值
sc = ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2], 
                c=direction, cmap='seismic', s=2, alpha=0.6)

plt.colorbar(sc, label='Direction of Movement (df/dt)')
ax.set_title(f"Neural Manifold Colored by Direction\n(Hardware: RTX 5070 Ti)", fontsize=15)

# 强制 x, y, z 轴的比例一致，消除视觉拉伸
ax.set_box_aspect([1, 1, 1]) 
# 调整视角为纯正的俯视图 (Top-down view)
# ax.view_init(elev=90, azim=0) 

# 移除坐标轴，增加科技感
ax.set_axis_off() 

plt.show()

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import cebra
import numpy as np
import matplotlib.pyplot as plt

# 1. 准备画布 (3行6列，对应 18 个 Session)
fig = plt.figure(figsize=(24, 12))
plt.suptitle("Universal $S^1$ Topology Across 18 Sessions (A1 Auditory Cortex)", fontsize=24, y=0.98)

device = 'cuda'
target_points = 5000  # 每个 Session 提取的控制点数

print("🚀 5070 Ti 正在批量渲染 18 个拓扑环，预计需要 2-3 分钟，请稍候...")

# 2. 循环遍历 18 个 Session
for session_idx in range(18):
    print(f"⏳ 正在计算 Session {session_idx}/17 ...")
    
    # 🌟 修复点 1：正确动态提取当前 Session 的数据
    r_full = n_data_s[session_idx].T
    pos_full = f_data_s[session_idx]['Position'].values
    
    # 🌟 修复点 2：使用“均匀降采样”代替“截取前5000点”
    # 这样可以保证涵盖整个 30 分钟实验的所有角度，画出完整的环！
    step = max(1, len(pos_full) // target_points) 
    r_sub = r_full[::step].astype(np.float32)
    pos_sub = pos_full[::step].astype(np.float32)
    
    # 清洗掉可能存在的 NaN 值
    valid_mask = ~np.isnan(pos_sub)
    r_valid = r_sub[valid_mask]
    pos_valid = pos_sub[valid_mask]
    
    # 🌟 修复点 3：找回 L2 归一化（洗掉幅值噪音，逼迫流形投影到球面上）
    norms = np.linalg.norm(r_valid, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    r_norm = r_valid / norms

    # 3. 训练当前 Session 的 CEBRA 模型
    model = cebra.CEBRA(
        model_architecture='offset10-model',
        batch_size=4096,
        learning_rate=3e-4,
        output_dimension=3,
        max_iterations=2000, # 5000 个点用 2000 次迭代足够收敛
        distance='cosine',
        conditional='time_delta',
        device=device
    )
    
    model.fit(r_norm, pos_valid)
    embedding = model.transform(r_norm)
    
    # 4. 绘制当前 Session 的 3D 流形
    ax = fig.add_subplot(3, 6, session_idx + 1, projection='3d')
    sc = ax.scatter(embedding[:, 0], embedding[:, 1], embedding[:, 2], 
                    c=pos_valid, cmap='twilight_shifted', s=2, alpha=0.6)
    
    ax.set_title(f"Session {session_idx}", fontsize=14)
    ax.set_axis_off()
    
    # 强制统一坐标轴比例为 1:1:1，防止甜甜圈被拉长变成椭圆
    all_max = np.max(np.abs(embedding), axis=0)
    ax.set_xlim(-all_max[0], all_max[0])
    ax.set_ylim(-all_max[1], all_max[1])
    ax.set_zlim(-all_max[2], all_max[2])
    ax.set_box_aspect([1, 1, 1])

# 整体排版并显示
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
print("✅ 18 宫格渲染完成！")

In [ ]:
# 1. Setup Parameters
t_window_pre = 0.32
t_window_post = 0.22

n_bins_pre = int(t_window_pre / dt)
n_bins_post = int(t_window_post / dt)
expected_length = n_bins_pre + n_bins_post


In [ ]:
for n_data, f_data in zip(n_data_s, f_data_s):
    print(f_data.groupby(['Block', 'Condition', 'Frequency_changes']).size())

In [ ]:
for n_data, f_data in zip(n_data_s, f_data_s):
    print(f_data[(f_data['Block'] == 5) & (f_data['Condition'] == 1) & (f_data['Frequency_changes'] == 1)].shape[0])

In [ ]:
import os

base_path = r"C:\Users\PenPen\Desktop\Ferret\Data\Bohan"

all_sessions_path = [
    os.path.join(base_path, name) 
    for name in os.listdir(base_path) 
    if os.path.isdir(os.path.join(base_path, name))
]

all_sessions_path[:2]